# Week 1b · M: learn how the world moves

Encode every frame with V, then train an LSTM with a mixture-density head to predict the next z from the current z and action. This M also predicts reward and episode end, which is what lets the controller train inside the dream in the next notebook.

About 20 minutes on a T4.

In [ ]:
# --- Setup: runs on Colab or locally (from the notebooks/ folder) -----------
REPO_URL = "https://github.com/vamsi1905/world-models-lab.git"   # <- edit once
USE_DRIVE = True        # keep data and checkpoints on Google Drive across disconnects

import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
SMOKE = os.environ.get("WM_SMOKE") == "1"      # tiny sizes, used only to test the notebook
if IN_COLAB:
    if not os.path.exists("/content/world-models-lab"):
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/world-models-lab"], check=True)
    os.chdir("/content/world-models-lab")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "swig"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        WORK = "/content/drive/MyDrive/world-models-lab"
    else:
        WORK = "/content/work"
else:
    sys.path.insert(0, os.path.abspath(".."))
    WORK = os.path.abspath(os.environ.get("WM_WORK", "../work"))
os.makedirs(WORK, exist_ok=True)

import numpy as np, torch, matplotlib.pyplot as plt
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| work dir:", WORK, "| smoke test:", SMOKE)

In [ ]:
from wmlab.carracing.rollouts import load_rollouts
from wmlab.carracing.vae import ConvVAE, encode_rollouts
from wmlab.carracing.mdnrnn import train_mdnrnn, MDNRNN, sample_mdn
R = load_rollouts(f"{WORK}/carracing_random")
vae = ConvVAE().to(DEVICE); vae.load_state_dict(torch.load(f"{WORK}/vae.pt", map_location=DEVICE))
encode_rollouts(vae, R, device=DEVICE)
for r in R: del r["obs"]                                   # frames no longer needed; free the RAM
np.savez(f"{WORK}/latents.npz", **{f"{k}_{i}": r[k] for i, r in enumerate(R) for k in ("mu", "logvar", "act", "rew", "done")})
print(len(R), "rollouts encoded")

In [ ]:
MDN_CKPT = f"{WORK}/mdnrnn.pt"
if os.path.exists(MDN_CKPT):
    mdn = MDNRNN().to(DEVICE); mdn.load_state_dict(torch.load(MDN_CKPT, map_location=DEVICE)); hist = []
else:
    mdn, hist = train_mdnrnn(R, epochs=3 if SMOKE else 40, seq_len=32 if SMOKE else 100,
                             batch_size=8 if SMOKE else 64, device=DEVICE, log_every=200)
    torch.save(mdn.state_dict(), MDN_CKPT)
if hist:
    h = np.array(hist); fig, ax = plt.subplots(1, 2, figsize=(10, 3))
    ax[0].plot(h[:, 1], lw=.5); ax[0].set_title("next-z negative log-likelihood")
    ax[1].plot(h[:, 2], lw=.5); ax[1].set_title("reward MSE"); ax[1].set_yscale("log"); plt.show()

## Dream versus reality, open loop
Take a real rollout. Give M the first real frame and then only the *recorded actions*, and let it imagine the rest. Decode both. Early frames should match; later ones drift, then blur. That blur is the averaging a Gaussian mixture does when it is unsure. It is precisely the weakness that pushed DreamerV2 to discrete latents, so note where it sets in.

In [ ]:
from wmlab.carracing.rollouts import load_rollouts
real = load_rollouts(f"{WORK}/carracing_random", limit=1)[0]
T = min(60, len(real["act"]) - 1)
with torch.no_grad():
    z = torch.from_numpy(R[0]["mu"][:1]).to(DEVICE); state = None; dream = [z]
    for t in range(T):
        a = torch.from_numpy(real["act"][t:t+1]).to(DEVICE)
        log_pi, mu, ls, *_ , state = mdn(z[:, None], a[:, None], state)
        z = sample_mdn(log_pi[:, 0], mu[:, 0], ls[:, 0], temperature=1.0); dream.append(z)
    imgs = vae.decode(torch.cat(dream)).permute(0, 2, 3, 1).cpu().numpy()
steps = np.linspace(0, T, 8).astype(int)
fig, ax = plt.subplots(2, 8, figsize=(14, 3.8))
for j, s in enumerate(steps):
    ax[0, j].imshow(real["obs"][s]); ax[1, j].imshow(imgs[s]); ax[0, j].set_title(f"t={s}", fontsize=8)
    ax[0, j].axis("off"); ax[1, j].axis("off")
ax[1, 0].set_title("dream", loc="left", fontsize=8)
plt.savefig(f"{WORK}/open_loop_dream.png", dpi=150, bbox_inches="tight"); plt.show()